<a href="https://colab.research.google.com/github/shreyashan-git/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop_tarts = cat[cat['product_name'].str.lower().str.contains('pop')]

print('Pop-Tart rows:', len(pop_tarts))
display(pop_tarts)

Pop-Tart rows: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


This code filters the dataset to find all products with "pop" in their name, regardless of capitalization. It then prints the number of matching rows and displays those products.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = cat[
    cat['product_name'].str.lower().str.contains('strawberry')
]

print(strawberry[['sku', 'product_name']].to_string(index=False))

          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry


This code filters the dataset to find all products with "strawberry" in their name, regardless of capitalization. It then prints the SKUs and product names of all matching products.

**Pop-Tart rows that are a different product:** Pop-Tarts Blueberry, this is a different flavor and should stay separate from strawberry Pop-Tarts.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
sku_map = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW'
}

cat['canonical_sku'] = cat['sku'].replace(sku_map)

display(cat)

,sku,product_name,category,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,POPTART-STRAW


This code maps the different strawberry poptart SKUs to one standard SKU, while keeping all other SKUs unchanged. It then creates a new canonical_sku column and displays the updated dataset.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
cat['category'] = cat['category'].replace({
    'Breakfast Foods': 'Breakfast'
})

print(cat['category'].value_counts())

category
Breakfast    3
Name: count, dtype: int64


This code combines "Breakfast Foods" and "Breakfast" into one category called "Breakfast" to keep the labels consistent. It then prints the number of products in each category.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re

def normalize_name(name):
    name = name.lower()
    name = re.sub(r'[^\w\s]', ' ', name)
    return ' '.join(sorted(name.split()))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

groups = cat.groupby('normalized_name').agg(
    sku_count=('sku', 'nunique'),
    skus=('sku', list)
)

duplicates = groups[groups['sku_count'] > 1]

display(duplicates)


,sku_count,skus
normalized_name,,
pop strawberry tarts,2,"[PT-12, POPTART-STRAW]"


This code normalizes product names by making them lowercase, removing punctuation, and sorting the words to identify possible duplicates. It then groups products with matching normalized names and displays those with more than one unique SKU.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
original_rows = len(cat)

cat['canonical_sku'] = cat['sku'].replace(sku_map)

assert cat['canonical_sku'].notna().all()
assert len(cat) == original_rows

print("Mapping is safe!")
print("Rows:", len(cat))
print("Nulls:", cat['canonical_sku'].isna().sum())

Mapping is safe!
Rows: 3
Nulls: 0


This code checks that mapping the SKUs did not create any missing values or change the number of rows in the dataset. It then prints a confirmation that the mapping is safe, along with the total number of rows and missing values.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) So basically, I only had poptarts in this dataset, but with a full catalog, I would look for products with the similar names and different SKUs.
b) So basically, consolidation becomes a business decision when two products look similar but might actually be different. For example, I wouldn't be sure whether two poptart products with slightly different names are the same item. I would ask the product manager or inventory team before combining them.